# 03_rectified_lpjepa.ipynb — Rectified LpJEPA objectives

**Goal**  
Compare this notebook's variance/covariance and L1/entropy proxy with the paper's Rectified Laplace RDMReg objective.

**Paper result**  
On ImageNet-100, the paper reports 84.72% encoder top-1 accuracy and 0.6940 L0 sparsity for its Rectified Laplace setting. Its LeJEPA baseline reports 84.80% accuracy and 1.0000 L0 sparsity. These are paper results, not local targets.

**Local study**  
Use CIFAR-10, a roughly 455,000-parameter CNN plus a 256-wide, three-layer projector, 640 unlabeled images, 20 epochs, 512 projections, and five training seeds. Both arms use the same nonnegative projector output. The paper-aligned loss uses weights 25/125, `p=1`, `mu=0`, and scale `1/sqrt(2)`. The paper's ImageNet-100 run uses a ResNet-50, a 2,048-wide projector, 1,000 epochs, LARS, and 8,192 projections.

This is a local objective study, not an ImageNet benchmark reproduction. The current proxy outputs are not the paper's RDMReg.

**Sources**  
- [Rectified LpJEPA paper, version 2](https://arxiv.org/html/2602.01456)  
- [Rectified LpJEPA official code](https://github.com/YilunKuang/rectified-lp-jepa)  
- [CIFAR-10 official data page](https://www.cs.toronto.edu/~kriz/cifar.html)

In [ ]:
import json
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from scipy.stats import t as student_t
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CIFAR10

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial import linear_probe_accuracy, sigreg_loss
from jepa_tutorial.rdmreg import rectified_distribution_matching_loss
from jepa_tutorial.paper_losses import rectified_laplace_rdmreg_loss, sample_rectified_laplace

device = torch.device("cpu")
torch.set_num_threads(min(8, torch.get_num_threads()))
BATCH_SIZE = 128
NUM_EPOCHS = 20
DATA_SEED = 20261014
TRAIN_SEEDS = [7, 17, 27, 37, 47]
print({"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__, "numpy": np.__version__, "scipy": scipy.__version__, "device": str(device), "torch_threads": torch.get_num_threads()})

In [ ]:
# Small deterministic checks for the paper-aligned objective.
probe = torch.rand(32, 16, generator=torch.Generator().manual_seed(303), requires_grad=True)
target_a = sample_rectified_laplace(
    (32, 16), device=device, dtype=probe.dtype,
    generator=torch.Generator().manual_seed(404),
)
target_b = sample_rectified_laplace(
    (32, 16), device=device, dtype=probe.dtype,
    generator=torch.Generator().manual_seed(404),
)
assert target_a.shape == probe.shape and torch.isfinite(target_a).all()
assert (target_a >= 0).all(), "Rectified Laplace targets must be nonnegative"
assert torch.equal(target_a, target_b), "Target sampling must repeat with the same seed"
loss_a = rectified_laplace_rdmreg_loss(
    probe, num_projections=16, generator=torch.Generator().manual_seed(505),
)
loss_b = rectified_laplace_rdmreg_loss(
    probe, num_projections=16, generator=torch.Generator().manual_seed(505),
)
assert loss_a.ndim == 0 and torch.isfinite(loss_a)
assert torch.equal(loss_a, loss_b), "RDMReg must repeat with the same generator seed"
loss_a.backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
try:
    rectified_laplace_rdmreg_loss(torch.full((8, 4), -1.0))
except ValueError:
    pass
else:
    raise AssertionError("RDMReg must reject negative projector output")
print("RDMReg shape, finiteness, target range, gradient, and repeatability checks passed")


In [ ]:
DATA_ROOT = repo_root / "notebooks" / "data"
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2430, 0.2610)
normalize = T.Normalize(MEAN, STD)
train_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    normalize,
])
eval_transform = T.Compose([T.ToTensor(), normalize])

try:
    train_base = CIFAR10(root=str(DATA_ROOT), train=True, download=True)
    test_base = CIFAR10(root=str(DATA_ROOT), train=False, download=True)
except Exception as exc:
    raise RuntimeError("Could not load CIFAR-10. See https://www.cs.toronto.edu/~kriz/cifar.html") from exc


def balanced_indices(labels, candidates, per_class, rng):
    labels = np.asarray(labels)
    candidates = np.asarray(candidates, dtype=np.int64)
    selected = []
    for class_id in range(10):
        pool = candidates[labels[candidates] == class_id]
        if len(pool) < per_class:
            raise ValueError(f"Not enough CIFAR-10 examples for class {class_id}")
        selected.extend(rng.choice(pool, per_class, replace=False).tolist())
    return np.asarray(sorted(selected), dtype=np.int64)


split_rng = np.random.default_rng(DATA_SEED)
all_train_indices = np.arange(len(train_base))
ssl_indices = balanced_indices(train_base.targets, all_train_indices, 64, split_rng)
probe_candidates = np.setdiff1d(all_train_indices, ssl_indices)
probe_indices = balanced_indices(train_base.targets, probe_candidates, 100, split_rng)
test_indices = balanced_indices(test_base.targets, np.arange(len(test_base)), 100, split_rng)


class TwoViewSubset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, _ = self.dataset[int(self.indices[position])]
        return train_transform(image), train_transform(image)


class LabeledSubset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, label = self.dataset[int(self.indices[position])]
        return eval_transform(image), int(label)


probe_loader = DataLoader(LabeledSubset(train_base, probe_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(LabeledSubset(test_base, test_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print({"ssl_images": len(ssl_indices), "probe_images": len(probe_indices), "test_images": len(test_indices), "data_seed": DATA_SEED})

In [ ]:
class SmallEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.embedding = nn.Linear(256, embed_dim)

    def forward(self, images):
        return self.embedding(self.features(images).flatten(1))


class RectifiedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = SmallEncoder()
        self.projector = nn.Sequential(
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )

    def forward(self, images):
        features = self.encoder(images)
        return features, self.projector(features)


model_parameters = sum(p.numel() for p in RectifiedModel().parameters())
print({"total_parameters": model_parameters, "embedding_dim": 256, "projector_dim": 256})

In [ ]:
def collect_features(model, loader):
    model.eval()
    encoder_features, projector_features, labels = [], [], []
    with torch.no_grad():
        for images, targets in loader:
            features, projected = model(images.to(device))
            encoder_features.append(features.cpu().numpy())
            projector_features.append(projected.cpu().numpy())
            labels.append(targets.numpy())
    return np.concatenate(encoder_features), np.concatenate(projector_features), np.concatenate(labels)


def run_method(method, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = RectifiedModel().to(device)
    train_generator = torch.Generator().manual_seed(seed + 1000)
    loss_generator = torch.Generator().manual_seed(seed + 2000)
    loader = DataLoader(
        TwoViewSubset(train_base, ssl_indices),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator,
        num_workers=0,
    )
    optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-5)
    started = time.perf_counter()
    final_loss = float("nan")

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_losses = []
        for view1, view2 in loader:
            view1, view2 = view1.to(device), view2.to(device)
            feature1, z1 = model(view1)
            feature2, z2 = model(view2)
            invariance = F.mse_loss(z1, z2)
            if method == "teaching_proxy":
                sig_proxy = 0.5 * (sigreg_loss(z1) + sigreg_loss(z2))
                rdm_proxy = 0.5 * (
                    rectified_distribution_matching_loss(z1)
                    + rectified_distribution_matching_loss(z2)
                )
                loss = invariance + 0.08 * sig_proxy + 0.02 * rdm_proxy
            else:
                rdm = 0.5 * (
                    rectified_laplace_rdmreg_loss(
                        z1, num_projections=512, generator=loss_generator
                    )
                    + rectified_laplace_rdmreg_loss(
                        z2, num_projections=512, generator=loss_generator
                    )
                )
                loss = 25.0 * invariance + 125.0 * rdm
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch + 1}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach()))
        final_loss = float(np.mean(epoch_losses))

    probe_encoder, probe_projector, probe_labels = collect_features(model, probe_loader)
    test_encoder, test_projector, test_labels = collect_features(model, test_loader)
    encoder_acc = linear_probe_accuracy(probe_encoder, probe_labels, test_encoder, test_labels)
    projector_acc = linear_probe_accuracy(probe_projector, probe_labels, test_projector, test_labels)
    norms = np.linalg.norm(test_projector, axis=1)
    l1_per_sample = np.zeros_like(norms)
    nonzero = norms > 1e-12
    l1_per_sample[nonzero] = (
        np.abs(test_projector[nonzero]).sum(axis=1) / norms[nonzero]
    ) ** 2 / test_projector.shape[1]
    l0 = (test_projector != 0).mean(axis=1)
    return {
        "method": method,
        "seed": seed,
        "status": "completed",
        "encoder_probe_top1": encoder_acc,
        "projector_probe_top1": projector_acc,
        "projector_l1_sparsity": float(l1_per_sample.mean()),
        "projector_l0_sparsity": float(l0.mean()),
        "projector_zero_vector_fraction": float((~nonzero).mean()),
        "final_train_loss": final_loss,
        "elapsed_seconds": time.perf_counter() - started,
    }


METHODS = ["teaching_proxy", "paper_aligned_rdmreg"]
RUNS = []
for method in METHODS:
    for seed in TRAIN_SEEDS:
        try:
            record = run_method(method, seed)
        except Exception as exc:
            record = {"method": method, "seed": seed, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
        RUNS.append(record)
        print({"method": method, "seed": seed, "status": record["status"], "projector_probe_top1": record.get("projector_probe_top1"), "projector_l0_sparsity": record.get("projector_l0_sparsity"), "elapsed_seconds": record.get("elapsed_seconds"), "error": record.get("error")})

In [ ]:
def summarize(values):
    values = np.asarray(values, dtype=float)
    if not len(values):
        return {"n": 0, "mean": None, "median": None, "ci95": None}
    if len(values) < 2:
        ci = None
    else:
        half = student_t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))
        ci = [float(values.mean() - half), float(values.mean() + half)]
    return {"n": len(values), "mean": float(values.mean()), "median": float(np.median(values)), "ci95": ci, "interval": "two-sided Student-t; uncertain at n=5"}


SUMMARY = {}
metric_names = (
    "encoder_probe_top1",
    "projector_probe_top1",
    "projector_l1_sparsity",
    "projector_l0_sparsity",
)
for method in METHODS:
    selected = [r for r in RUNS if r["method"] == method]
    SUMMARY[method] = {
        "runs": selected,
        "metrics": {
            name: summarize([r[name] for r in selected if r["status"] == "completed" and name in r])
            for name in metric_names
        },
    }
paired_deltas = {}
for metric in metric_names:
    deltas = []
    for seed in TRAIN_SEEDS:
        by_method = {
            run["method"]: run for run in RUNS
            if run["seed"] == seed and run["status"] == "completed"
        }
        if all(method in by_method for method in METHODS):
            deltas.append(by_method["paper_aligned_rdmreg"][metric] - by_method["teaching_proxy"][metric])
    paired_deltas[metric] = summarize(deltas)
    paired_deltas[metric]["paper_aligned_minus_teaching"] = True
    paired_deltas[metric]["interval"] = "paired two-sided Student-t; uncertain at n=5"

print(json.dumps({"settings": {"data_seed": DATA_SEED, "training_seeds": TRAIN_SEEDS, "ssl_images": len(ssl_indices), "probe_images": len(probe_indices), "test_images": len(test_indices), "total_parameters": model_parameters, "batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS, "learning_rate": 0.003, "weight_decay": 1e-5, "augmented_views": 2, "projector_width": 256, "projector_layers": 3, "rdm_projection_count": 512, "invariance_weight": 25.0, "rdmreg_weight": 125.0}, "runs": RUNS, "summary": SUMMARY, "paired_deltas": paired_deltas}, indent=2))

## Paper reproduction boundary

The paper's ImageNet-100 result is paper-reported. This local CIFAR-10 study uses a smaller network, smaller labeled and unlabeled splits, fewer epochs, AdamW instead of LARS, and 512 rather than 8,192 projections. A low sparsity score is not useful evidence by itself; read it beside both linear-probe scores and the zero-vector fraction.

## Local run notes

The previous saved output in this notebook reported chance-level probe accuracy and near-zero feature variance, with DataLoader shutdown warnings. It came from the old, unpaired notebook path. The new clean-kernel comparison records its own runs and does not treat those saved outputs as evidence.

## Suggested exercises

1. Change the target mean shift and report accuracy with both sparsity measures.  
2. Increase the projection count and compare runtime.  
3. Read the paper's ImageNet-100 setup beside the local settings in the run report.